# Assignment Beta, Question 4: Evaluating an LLM sentiment classifier
Student ID: 23K-0001

Confusion matrices, classification metrics, prompt stability, text-generation metrics and perplexity.

In [1]:
import math
import numpy as np
import pandas as pd
from collections import Counter

classes = ["Positive", "Negative", "Neutral"]
cm1 = np.array([[100, 5, 15], [8, 80, 12], [20, 10, 50]])   # Stage 1, rows = actual
cm2 = np.array([[106, 4, 10], [6, 84, 10], [9, 7, 64]])     # Stage 2

## 1. Accuracy and 95% confidence interval (Stage 1)

In [2]:
def accuracy_ci(cm, z=1.96):
    n = cm.sum()
    acc = np.trace(cm) / n
    se = math.sqrt(acc * (1 - acc) / n)
    wald = (acc - z * se, acc + z * se)
    den = 1 + z**2 / n
    centre = (acc + z**2 / (2 * n)) / den
    half = z * math.sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2)) / den
    return acc, se, wald, (centre - half, centre + half)

acc, se, wald, wilson = accuracy_ci(cm1)
print(f"Accuracy = {np.trace(cm1)}/{cm1.sum()} = {acc:.4f}")
print(f"SE = {se:.4f}")
print(f"95% CI (Wald)   = [{wald[0]:.4f}, {wald[1]:.4f}]")
print(f"95% CI (Wilson) = [{wilson[0]:.4f}, {wilson[1]:.4f}]")

Accuracy = 230/300 = 0.7667
SE = 0.0244
95% CI (Wald)   = [0.7188, 0.8145]
95% CI (Wilson) = [0.7156, 0.8110]


## 2. Precision, recall and F1 per class (Stage 1)

In [3]:
def per_class(cm):
    tp = np.diag(cm).astype(float)
    prec = tp / cm.sum(axis=0)
    rec = tp / cm.sum(axis=1)
    f1 = 2 * prec * rec / (prec + rec)
    return pd.DataFrame({"TP": tp.astype(int), "Column total": cm.sum(axis=0), "Row total": cm.sum(axis=1),
                         "Precision": prec.round(4), "Recall": rec.round(4), "F1": f1.round(4)}, index=classes)

df1 = per_class(cm1)
df1

,TP,Column total,Row total,Precision,Recall,F1
Positive,100,128,120,0.7812,0.8333,0.8065
Negative,80,95,100,0.8421,0.8000,0.8205
Neutral,50,77,80,0.6494,0.6250,0.6369


## 3. Macro, weighted and micro F1 (Stage 1)

In [4]:
def averages(cm):
    df = per_class(cm)
    support = cm.sum(axis=1)
    macro = df["F1"].mean()
    weighted = (df["F1"] * support).sum() / support.sum()
    micro = np.trace(cm) / cm.sum()      # micro P = R = F1 = accuracy for single-label multiclass
    return macro, weighted, micro

m1 = averages(cm1)
print(f"Macro F1    = {m1[0]:.4f}")
print(f"Weighted F1 = {m1[1]:.4f}")
print(f"Micro F1    = {m1[2]:.4f}")

Macro F1    = 0.7546
Weighted F1 = 0.7659
Micro F1    = 0.7667


## 4. Stage 1 error analysis
Off-diagonal cells show where the model is wrong. Most errors involve the Neutral class.

In [5]:
errors = []
for i, a in enumerate(classes):
    for j, p in enumerate(classes):
        if i != j:
            errors.append((f"{a} predicted as {p}", int(cm1[i, j])))
err = pd.DataFrame(errors, columns=["Confusion", "Count"]).sort_values("Count", ascending=False)
err["Share of errors"] = (err["Count"] / err["Count"].sum()).round(3)
neutral_errors = err[err["Confusion"].str.contains("Neutral")]["Count"].sum()
print(f"Total errors: {err['Count'].sum()}, involving Neutral: {neutral_errors}")
err

Total errors: 70, involving Neutral: 57


,Confusion,Count,Share of errors
4,Neutral predicted as Positive,20,0.286
1,Positive predicted as Neutral,15,0.214
3,Negative predicted as Neutral,12,0.171
5,Neutral predicted as Negative,10,0.143
2,Negative predicted as Positive,8,0.114
0,Positive predicted as Negative,5,0.071


## 5. Stage 2 re-evaluation and comparison

In [6]:
acc2, se2, wald2, wilson2 = accuracy_ci(cm2)
print(f"Stage 2 accuracy = {acc2:.4f}, 95% CI (Wald) = [{wald2[0]:.4f}, {wald2[1]:.4f}]")
df2 = per_class(cm2)
display(df2)

m2 = averages(cm2)
comparison = pd.DataFrame({
    "Stage 1": [acc, m1[0], m1[1], m1[2]],
    "Stage 2": [acc2, m2[0], m2[1], m2[2]],
}, index=["Accuracy", "Macro F1", "Weighted F1", "Micro F1"])
comparison["Change"] = comparison["Stage 2"] - comparison["Stage 1"]
comparison.round(4)

Stage 2 accuracy = 0.8467, 95% CI (Wald) = [0.8059, 0.8874]


,TP,Column total,Row total,Precision,Recall,F1
Positive,106,121,120,0.8760,0.8833,0.8797
Negative,84,95,100,0.8842,0.8400,0.8615
Neutral,64,84,80,0.7619,0.8000,0.7805


,Stage 1,Stage 2,Change
Accuracy,0.7667,0.8467,0.0800
Macro F1,0.7546,0.8406,0.0859
Weighted F1,0.7659,0.8472,0.0812
Micro F1,0.7667,0.8467,0.0800


## 6. Prompt stability over 5 example orderings

In [7]:
accs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
print(f"Mean = {accs.mean():.4f}")
print(f"Sample SD (n-1) = {accs.std(ddof=1):.4f}")
print(f"Population SD (n) = {accs.std(ddof=0):.4f}")
print(f"Range = {accs.min():.3f} to {accs.max():.3f}")
print(f"Coefficient of variation = {accs.std(ddof=1) / accs.mean() * 100:.2f}%")

Mean = 0.8454
Sample SD (n-1) = 0.0067
Population SD (n) = 0.0060
Range = 0.837 to 0.853
Coefficient of variation = 0.80%


## 7. Generated justification: BLEU-4, ROUGE-1/2/L and METEOR

In [8]:
ref = "the delivery was late but the product quality is good".split()
cand = "the product quality is good but delivery was slow".split()

def ngrams(t, n):
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))

def clipped(c, r, n):
    cc, rc = ngrams(c, n), ngrams(r, n)
    return sum(min(v, rc[k]) for k, v in cc.items()), sum(cc.values()), sum(rc.values())

# BLEU-4
ps = []
for n in range(1, 5):
    m, tc, _ = clipped(cand, ref, n)
    ps.append(m / tc)
    print(f"p{n} = {m}/{tc} = {m / tc:.4f}")
bp = math.exp(1 - len(ref) / len(cand)) if len(cand) < len(ref) else 1.0
bleu4 = bp * math.exp(sum(math.log(p) for p in ps) / 4)
print(f"BP = {bp:.4f}, BLEU-4 = {bleu4:.4f}")

p1 = 8/9 = 0.8889
p2 = 5/8 = 0.6250
p3 = 3/7 = 0.4286
p4 = 2/6 = 0.3333
BP = 0.8948, BLEU-4 = 0.4750


In [9]:
# ROUGE-1 and ROUGE-2
for n in (1, 2):
    m, tc, tr = clipped(cand, ref, n)
    p, r = m / tc, m / tr
    print(f"ROUGE-{n}: matches={m}, P={p:.4f}, R={r:.4f}, F1={2 * p * r / (p + r):.4f}")

# ROUGE-L via longest common subsequence
def lcs(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

l = lcs(cand, ref)
p, r = l / len(cand), l / len(ref)
print(f"ROUGE-L: LCS={l}, P={p:.4f}, R={r:.4f}, F1={2 * p * r / (p + r):.4f}")

ROUGE-1: matches=8, P=0.8889, R=0.8000, F1=0.8421
ROUGE-2: matches=5, P=0.6250, R=0.5556, F1=0.5882
ROUGE-L: LCS=5, P=0.5556, R=0.5000, F1=0.5263


In [10]:
# METEOR, exact matches only: alpha = 0.9, beta = 3, gamma = 0.5
matches = 8          # all candidate words except "slow"
chunks = 3           # [the product quality is good] [but] [delivery was]
P, R = matches / len(cand), matches / len(ref)
fmean = P * R / (0.9 * R + 0.1 * P)
penalty = 0.5 * (chunks / matches) ** 3
meteor = fmean * (1 - penalty)
print(f"P={P:.4f}, R={R:.4f}, Fmean={fmean:.4f}, penalty={penalty:.4f}, METEOR={meteor:.4f}")

P=0.8889, R=0.8000, Fmean=0.8791, penalty=0.0264, METEOR=0.8559


## 8. Perplexity of the generated justification

In [11]:
tokens = cand
probs = [0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20]
lnp = [math.log(p) for p in probs]
print(pd.DataFrame({"token": tokens, "p": probs, "ln p": np.round(lnp, 4)}).to_string(index=False))
avg_nll = -sum(lnp) / len(probs)
print(f"Sum ln p = {sum(lnp):.4f}")
print(f"Average NLL = {avg_nll:.4f}")
print(f"Perplexity = exp({avg_nll:.4f}) = {math.exp(avg_nll):.4f}")

   token    p    ln p
     the 0.60 -0.5108
 product 0.40 -0.9163
 quality 0.50 -0.6931
      is 0.80 -0.2231
    good 0.70 -0.3567
     but 0.30 -1.2040
delivery 0.25 -1.3863
     was 0.50 -0.6931
    slow 0.20 -1.6094
Sum ln p = -7.5929
Average NLL = 0.8437
Perplexity = exp(0.8437) = 2.3249
